# G8 — Balayages finaux : vitesse, robustesse et prétraitement ciblé

Ce notebook récupère le code **depuis GitHub** (donc les fichiers de B : `frcnn.py`, `preprocess.py`), puis, pour **un modèle à la fois** :
1. mesure sa **vitesse** sur l'image propre (conf = 0,25) ;
2. lance le **balayage de robustesse** (6 dégradations × 5 niveaux) si les résultats n'existent pas encore ;
3. lance le **balayage avec prétraitement ciblé** (chaque prétraitement sur la dégradation qu'il vise).

**Réglages AVANT de lancer** (panneau de droite) :
- **Add Input > Notebooks** : `01_kaggle_data_prep`.
- **Add Input > Datasets** : les poids du modèle choisi (`g8-weights`, `g8-weights-rtdetr` ou `g8-weights-frcnn`).
- **Accelerator : GPU T4**, **Internet : On**.
- Pour comparer les vitesses, lancez les **3 modèles sur le même type de GPU** (la cellule 3 l'affiche).

Exécutez les cellules dans l'ordre. Pour changer de modèle : modifiez la cellule 1 et relancez tout.

In [1]:
MODEL = 'frcnn'      # 'yolo11' | 'rtdetr' | 'frcnn'
WEIGHTS_FILE = {'yolo11': 'yolo11_best.pt', 'rtdetr': 'rtdetr_best.pt', 'frcnn': 'frcnn_best.pth'}[MODEL]

RUN_SPEED = True         # mesure de vitesse sur l'image propre
RUN_SWEEP = True         # balayage 6 dégradations x 5 niveaux (mettre False pour yolo11 et rtdetr : déjà faits)
RUN_PREPROCESS = True    # balayage avec prétraitement ciblé

# prétraitement -> dégradation qu'il vise (un prétraitement est évalué sur son type de dégradation + l'image propre)
PLAN = {'night': ['gamma', 'clahe_soft'], 'blur': ['sharpen_soft'], 'fog': ['dehaze'], 'jpeg': ['denoise']}
print(MODEL, WEIGHTS_FILE)

frcnn frcnn_best.pth


## 2. Récupérer le code (GitHub)
Si le dépôt est **privé**, le clonage échoue : le notebook cherche alors le code dans un dataset Kaggle (téléchargez le dépôt en ZIP depuis GitHub : *Code > Download ZIP*, puis *Create > New Dataset*, et ajoutez-le en Input).

In [2]:
import glob, os, shutil, subprocess
REPO_URL = 'https://github.com/salmaaz29/g8-traffic-sign-detection.git'
REPO = '/kaggle/working/g8-traffic-sign-detection'
os.chdir('/kaggle/working')
if not os.path.exists(f'{REPO}/src/models/frcnn.py'):
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], capture_output=True, text=True)
    print(r.stdout, r.stderr)
if not os.path.exists(f'{REPO}/src/models/frcnn.py'):
    hits = glob.glob('/kaggle/input/**/src/models/frcnn.py', recursive=True)
    assert hits, "Code introuvable : dépôt privé ? Créez un dataset Kaggle à partir du ZIP du dépôt GitHub."
    root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(root, REPO, dirs_exist_ok=True)
for d in ['weights', 'results']:
    os.makedirs(f'{REPO}/{d}', exist_ok=True)
%cd {REPO}
print(sorted(os.listdir('.')))
print(os.path.exists('src/models/frcnn.py'), os.path.exists('src/preprocessing/preprocess.py'))   # attendu : True True

 Cloning into 'g8-traffic-sign-detection'...

/kaggle/working/g8-traffic-sign-detection
['.git', '.gitignore', '.streamlit', 'README.md', 'app.py', 'config.yaml', 'notebooks', 'reports', 'requirements.txt', 'results', 'scripts', 'splits', 'src', 'weights']
True True


## 3. Installation et GPU

In [4]:
!pip install -q -U ultralytics pycocotools
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU'
print('GPU :', gpu)
assert torch.cuda.is_available(), "Activez le GPU (Settings > Accelerator), sinon les vitesses sont fausses."
open('/kaggle/working/results_gpu_%s.txt' % MODEL, 'w').write(gpu)

GPU : Tesla T4


8

## 4. Données de test et poids

In [5]:
hits = glob.glob('/kaggle/input/**/gtsdb_4cls/coco/test.json', recursive=True)
assert hits, "Données introuvables : ajoutez la sortie de 01_kaggle_data_prepa en Input."
src = os.path.dirname(os.path.dirname(hits[0]))
dst = f'{REPO}/data/processed/gtsdb_4cls'
os.makedirs(dst, exist_ok=True)
shutil.copytree(f'{src}/coco', f'{dst}/coco', dirs_exist_ok=True)
shutil.copytree(f'{src}/images/test', f'{dst}/images/test', dirs_exist_ok=True)
print('test :', len(os.listdir(f'{dst}/images/test')), 'images')      # attendu : 90

w = glob.glob(f'/kaggle/input/**/{WEIGHTS_FILE}', recursive=True)
assert w, f"{WEIGHTS_FILE} introuvable : ajoutez le dataset de poids en Input."
shutil.copy(w[0], f'{REPO}/weights/{WEIGHTS_FILE}')
print('poids :', w[0])

test : 90 images
poids : /kaggle/input/datasets/minada43/g8-weights-frcnn/weights/frcnn_best.pth


## 5. Vitesse et métriques sur l'image propre (conf = 0,25)

In [6]:
if RUN_SPEED:
    !python -m src.evaluation.evaluate --model {MODEL} --weights weights/{WEIGHTS_FILE} --data-dir data/processed/gtsdb_4cls --split test --out /kaggle/working/results/speed_{MODEL}.csv

eval: 100%|█████████████████████████████████████| 90/90 [00:40<00:00,  2.21it/s]

frcnn | prétraitement=none | dégradation=aucune niveau=0 | 90 images
  map50        : 0.990
  map50_95     : 0.825
  precision    : 0.876
  recall       : 0.987
  mean_conf    : 0.928
  ms_per_image : 182.658
  fps          : 5.475
  (vitesse mesurée à conf=0.25 ; à conf=0.001 : 181.5 ms/image)
  AP50 par classe : {'prohibitory': 1.0, 'danger': 0.999, 'mandatory': 0.993, 'other': 0.969}


## 6. Balayage de robustesse (sans prétraitement)
Quelques minutes à plusieurs dizaines de minutes selon le modèle. Relançable : les combinaisons déjà faites sont sautées.

In [7]:
if RUN_SWEEP:
    !python scripts/run_robustness.py --model {MODEL} --weights weights/{WEIGHTS_FILE} --data-dir data/processed/gtsdb_4cls --preprocess none --out /kaggle/working/results/{MODEL}.csv

frcnn   none     clean      niv 0  mAP50=0.990  mAP50-95=0.825  P=0.876  R=0.987
frcnn   none     rain       niv 1  mAP50=0.985  mAP50-95=0.810  P=0.864  R=0.973
frcnn   none     rain       niv 2  mAP50=0.977  mAP50-95=0.795  P=0.917  R=0.960
frcnn   none     rain       niv 3  mAP50=0.853  mAP50-95=0.672  P=0.906  R=0.773
frcnn   none     rain       niv 4  mAP50=0.480  mAP50-95=0.347  P=0.967  R=0.393
frcnn   none     rain       niv 5  mAP50=0.167  mAP50-95=0.118  P=1.000  R=0.113
frcnn   none     fog        niv 1  mAP50=0.987  mAP50-95=0.817  P=0.880  R=0.980
frcnn   none     fog        niv 2  mAP50=0.982  mAP50-95=0.808  P=0.890  R=0.967
frcnn   none     fog        niv 3  mAP50=0.973  mAP50-95=0.788  P=0.899  R=0.953
frcnn   none     fog        niv 4  mAP50=0.959  mAP50-95=0.764  P=0.933  R=0.933
frcnn   none     fog        niv 5  mAP50=0.915  mAP50-95=0.720  P=0.923  R=0.880
frcnn   none     night      niv 1  mAP50=0.986  mAP50-95=0.802  P=0.849  R=0.973
frcnn   none     night      

## 7. Balayage avec prétraitement ciblé
Chaque prétraitement est évalué sur **sa** dégradation (niveaux 1 à 5) et sur l'image propre (pour voir s'il dégrade les images saines).

In [8]:
if RUN_PREPROCESS:
    for kind, methods in PLAN.items():
        pre = ' '.join(methods)
        print('=====', kind, '<-', pre)
        get_ipython().system(f"python scripts/run_robustness.py --model {MODEL} --weights weights/{WEIGHTS_FILE} --data-dir data/processed/gtsdb_4cls --preprocess {pre} --kinds {kind} --out /kaggle/working/results/{MODEL}_preprocess.csv")

===== night <- gamma clahe_soft
frcnn   gamma    clean      niv 0  mAP50=0.991  mAP50-95=0.824  P=0.881  R=0.987
frcnn   gamma    night      niv 1  mAP50=0.985  mAP50-95=0.798  P=0.858  R=0.967
frcnn   gamma    night      niv 2  mAP50=0.879  mAP50-95=0.685  P=0.828  R=0.833
frcnn   gamma    night      niv 3  mAP50=0.589  mAP50-95=0.401  P=0.818  R=0.540
frcnn   gamma    night      niv 4  mAP50=0.340  mAP50-95=0.208  P=0.875  R=0.280
frcnn   gamma    night      niv 5  mAP50=0.146  mAP50-95=0.072  P=0.696  R=0.107
frcnn   clahe_soft clean      niv 0  mAP50=0.992  mAP50-95=0.825  P=0.877  R=1.000
frcnn   clahe_soft night      niv 1  mAP50=0.989  mAP50-95=0.802  P=0.869  R=0.973
frcnn   clahe_soft night      niv 2  mAP50=0.929  mAP50-95=0.721  P=0.848  R=0.933
frcnn   clahe_soft night      niv 3  mAP50=0.717  mAP50-95=0.509  P=0.764  R=0.647
frcnn   clahe_soft night      niv 4  mAP50=0.425  mAP50-95=0.268  P=0.775  R=0.413
frcnn   clahe_soft night      niv 5  mAP50=0.234  mAP50-95=0.137  P

## 8. Résultats

In [9]:
import pandas as pd
for name in [f'speed_{MODEL}.csv', f'{MODEL}.csv', f'{MODEL}_preprocess.csv']:
    p = f'/kaggle/working/results/{name}'
    if os.path.exists(p):
        d = pd.read_csv(p)
        print('\n=====', name)
        if name.startswith('speed'):
            print(d[['model', 'map50', 'map50_95', 'precision', 'recall', 'mean_conf', 'ms_per_image', 'fps']].round(3).to_string(index=False))
        else:
            print(d.pivot_table(index=['preprocess', 'degradation'], columns='level', values='map50').round(3))


===== speed_frcnn.csv
model  map50  map50_95  precision  recall  mean_conf  ms_per_image   fps
frcnn   0.99     0.825      0.876   0.987      0.928       182.658 5.475

===== frcnn.csv
level                      0      1      2      3      4      5
preprocess degradation                                         
none       blur          NaN  0.943  0.886  0.608  0.371  0.169
           clean        0.99    NaN    NaN    NaN    NaN    NaN
           fog           NaN  0.987  0.982  0.973  0.959  0.915
           jpeg          NaN  0.968  0.951  0.889  0.707  0.352
           night         NaN  0.986  0.944  0.780  0.479  0.254
           occlusion     NaN  0.976  0.921  0.824  0.509  0.234
           rain          NaN  0.985  0.977  0.853  0.480  0.167

===== frcnn_preprocess.csv
level                         0      1      2      3      4      5
preprocess   degradation                                          
clahe_soft   clean        0.992    NaN    NaN    NaN    NaN    NaN
         

## 9. Sauvegarde
Téléchargez les CSV de `/kaggle/working/results/` et placez-les dans `results/<modèle>/` du dépôt